# Energy Data Pipeline for Global Emissions and Renewable Energy Analysis using Databricks


**Name:** SHANON MWANGI 

**Student Number:** S2561435

## Background

Global energy systems are a major driver of economic development, but they are also the largest contributors to greenhouse gas emissions, particularly carbon dioxide (CO₂). Over the past decades, rising industrialisation, population growth, and energy demand have led to a significant increase in global emissions, raising concerns about climate change and environmental sustainability.

In response, there has been a global shift towards renewable energy sources such as wind, solar, and hydroelectric power. These sources offer a cleaner alternative to fossil fuels and play a critical role in reducing carbon emissions. However, the pace and scale of renewable energy adoption vary significantly across regions due to differences in economic development, policy frameworks, and resource availability.

Understanding the relationship between energy consumption, carbon emissions, and renewable energy adoption is essential for evaluating progress towards sustainability goals. In particular, metrics such as **CO₂ emissions per capita** provide insight into the environmental impact of different countries, while **renewable energy share** highlights the extent to which cleaner energy sources are being integrated into energy systems.

This project aims to explore these dynamics through a data-driven approach, enabling comparison across countries and regions, and identifying patterns and trends in global energy transition.

## Overview

This notebook presents the design and implementation of a prototype data pipeline for analysing global energy consumption, carbon emissions, and renewable energy adoption using Databricks.

The pipeline follows a structured approach, beginning with raw data ingestion and progressing through data cleaning, transformation, storage, and analytical visualisation. Particular emphasis is placed on handling missing data, engineering meaningful metrics (such as CO₂ emissions per capita and renewable energy share), and producing insights that support understanding of global energy trends.

The primary objective is to transform raw, heterogeneous energy data into a clean, structured dataset that enables meaningful analysis of:
- Regional trends in carbon emissions  
- Global distribution of renewable energy adoption  
- The relationship between renewable energy usage and emissions  

This notebook demonstrates not only the technical implementation of a data pipeline, but also critical decision-making in data preparation and analysis to ensure reliability and interpretability of results.

## 1. Source Data Management

In this stage, the raw energy dataset is ingested into Databricks and prepared for processing within a distributed environment.

To ensure proper data organisation and scalability, the dataset was first uploaded into the **Databricks Unity Catalog**. A dedicated **catalog, schema, and volume** were created to manage the data in a structured and reproducible manner:

- A **catalog** was used to organise data assets at a high level  
- A **schema** was created to logically group related datasets  
- A **volume** was used to store the raw CSV file as an external data source  

This approach reflects best practices in data engineering by separating storage from computation and enabling consistent data access across the pipeline.

The dataset, stored in CSV format, was then loaded into a Spark DataFrame using PySpark. Schema inference was enabled to automatically detect column data types, allowing for efficient downstream transformations.


In [0]:
# Load dataset
file_path = "/Volumes/workspace/energy_schema/bdp_energy/Consolidated Dataset - Panel format.csv"

df = spark.read.csv(file_path, header=True, inferSchema=True)

# Preview data
display(df)

## 2. ETL Transformations

This stage focuses on transforming the raw dataset into a structured and analysis-ready format. The process involves both data cleaning and feature engineering, with particular attention given to handling inconsistencies and ensuring meaningful interpretation of values.

---

### 2.1 Data Cleaning

The dataset required careful cleaning due to the presence of duplicate records and ambiguous representations of missing data.

Firstly, duplicate entries were removed based on a composite key of **Country and Year**, ensuring that each observation uniquely represents a country-year pair. This is particularly important for panel data, where multiple records for the same entity and time period can distort analysis.

Secondly, it was observed that several numerical columns used **zero values to represent missing data**, rather than true measurements. For example, a value of zero in CO₂ emissions or energy consumption does not necessarily indicate the absence of emissions, but rather missing or unreported data.

To address this, zero values in key variables were replaced with NULL values. This prevents misleading calculations, particularly when deriving per capita metrics or ratios, and ensures that missing data is handled explicitly during analysis.

The columns adjusted include:
- CO₂ emissions (`co2_mtco2`)
- Oil, gas, and coal consumption (`oilcons_ej`, `gascons_ej`, `coalcons_ej`)
- Renewable energy consumption (`renewables_ej`)

This approach improves data integrity by distinguishing between true zero values and missing observations.

---

### 2.2 Feature Engineering

Following data cleaning, new variables were created to enable more meaningful analysis of global energy trends.

Rather than relying on absolute values, which can be misleading due to differences in country size and population, derived metrics were introduced:

- **CO₂ emissions per capita**: provides a normalized measure of environmental impact across countries  
- **Energy consumption per capita**: allows comparison of energy usage independent of population size  
- **Renewable energy share**: measures the proportion of renewable energy relative to fossil fuel consumption, offering insight into energy transition  

These transformations convert raw data into interpretable indicators, supporting comparative and trend-based analysis across countries and regions.

Overall, this stage ensures that the dataset is both clean and analytically meaningful, forming a strong foundation for subsequent querying and visualisation.

In [0]:
from pyspark.sql.functions import col, when

# Remove duplicate country-year records
df_clean = df.dropDuplicates(["Country", "Year"])

# Replace zero values (used as missing data) with NULL
columns_to_fix = [
    "co2_mtco2",
    "oilcons_ej",
    "gascons_ej",
    "coalcons_ej",
    "renewables_ej"
]

for c in columns_to_fix:
    df_clean = df_clean.withColumn(
        c,
        when((col(c) == 0) | (col(c).isNull()), None).otherwise(col(c))
    )

display(df_clean)

### 2.3 Feature Engineering and Derived Metrics

To enable meaningful analysis, key variables were transformed into derived metrics that better capture relationships between energy consumption, emissions, and population.

Raw values such as total emissions or energy consumption can be misleading when comparing countries of different sizes. Therefore, normalization techniques were applied to create comparable indicators.

The following features were engineered:

- **CO₂ emissions per capita (`co2_per_capita`)**  
  This metric normalises total emissions by population, allowing fair comparison of environmental impact across countries regardless of size.

- **Energy consumption per capita (`energy_per_capita`)**  
  This provides insight into how intensively energy is used within a population, enabling comparison of energy demand patterns across regions.

- **Renewable energy share (`renewables_share`)**  
  This metric represents the proportion of renewable energy relative to total fossil fuel consumption (oil, gas, and coal). It serves as an indicator of a country’s progress towards transitioning to cleaner energy sources.

To ensure robustness, the `try_divide` function was used in all calculations. This prevents division-by-zero errors and ensures that invalid operations return NULL values instead of causing pipeline failure. This is particularly important given the presence of missing or zero values in the dataset.

Overall, these transformations convert raw data into interpretable indicators that support cross-country comparison, trend analysis, and evaluation of the relationship between renewable energy adoption and emissions.

In [0]:
from pyspark.sql.functions import expr

# Create derived analytical metrics
df_transformed = df_clean.withColumn(
    "co2_per_capita",
    expr("try_divide(co2_mtco2, pop)")
).withColumn(
    "energy_per_capita",
    expr("try_divide(oilcons_ej, pop)")
).withColumn(
    "renewables_share",
    expr("""
        try_divide(
            renewables_ej,
            oilcons_ej + gascons_ej + coalcons_ej
        )
    """)
)

# Preview transformed dataset
display(
    df_transformed.select("Country", "Year", "Region", "co2_per_capita", "renewables_share", "energy_per_capita")
    .orderBy("Country", "Year")
)

## 3. Data Storage for Analytics

Following transformation, the dataset was stored in **Delta Lake format** within the Databricks environment. This step represents the transition from raw and processed data into a structured, analytics-ready dataset.

Delta format was chosen due to its advantages over traditional file formats such as CSV, including:
- **Efficient querying** through optimized storage and indexing  
- **ACID transaction support**, ensuring data reliability and consistency  
- **Scalability**, enabling handling of large datasets in distributed environments  

The transformed dataset (`df_transformed`) was written to a designated path within the Databricks volume. This ensures persistence of the processed data and allows it to be reused across different stages of the pipeline without recomputation.

To demonstrate data retrieval and support reproducibility, the stored dataset was then reloaded into a new DataFrame (`df_final`). This simulates the process of accessing data from a storage layer into an analytical layer, which is a key component of modern data pipelines.

This separation between transformation and storage enhances modularity, allowing the pipeline to be extended or reused for further analysis.

In [0]:
# Save transformed data in Delta format (analytics-ready storage layer)
df_transformed.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/energy_schema/bdp_energy/energy_delta"
)

# Reload stored data for analysis
df_final = spark.read.format("delta").load(
    "/Volumes/workspace/energy_schema/bdp_energy/energy_delta"
)

## 4. Data Validation and Exploration

After storing and reloading the transformed dataset, an initial validation step was performed to ensure that the data was correctly persisted and remains consistent for analysis.

The dataset was inspected by selecting key variables, including country, year, region, and the derived metrics. Sorting the data by country and year allows for easier verification of temporal consistency and ensures that the panel structure of the dataset is preserved.

This step serves two important purposes:
- **Validation**: confirming that the transformation and storage processes were successful  
- **Exploration**: providing an overview of the dataset prior to analytical querying and visualisation  

By examining the structured dataset at this stage, potential anomalies such as missing values or unexpected patterns can be identified early, improving the reliability of subsequent analysis.

In [0]:
display(
    df_final
    .select("Country", "Year", "Region", "co2_per_capita", "renewables_share", "energy_per_capita")
    .orderBy("Country", "Year")
)

## 5. Data Preparation for Analysis

Before conducting analysis, the dataset was further refined to ensure that only relevant and reliable observations were included. This step focuses on preparing a clean analytical dataset by applying targeted filtering based on data quality and analytical requirements.

Several filtering decisions were made:

- **Removal of missing values**  
  Rows with NULL values in key variables (`co2_per_capita` and `renewables_share`) were excluded. This ensures that all subsequent analysis is based on complete and meaningful data, avoiding misleading interpretations.

- **Temporal filtering (Year ≥ 2010)**  
  The analysis was restricted to data from 2010 onwards. Earlier years contained a higher proportion of missing or incomplete records, which could distort trends and comparisons. Focusing on recent data improves reliability and reflects more current energy dynamics.

- **Exclusion of aggregated entries**  
  Entries such as *“Total Europe”*, *“Other”*, and *“Rest of World”* were removed. These represent aggregated or grouped data rather than individual countries. Including them would lead to duplication and bias when performing regional or country-level analysis.

This filtering process ensures that the resulting dataset (`df_analysis`) is:
- consistent  
- representative of real entities (countries)  
- suitable for comparative and trend-based analysis  

By explicitly defining these criteria, the analysis remains transparent and reproducible.

In [0]:
from pyspark.sql.functions import col

df_analysis = df_final.filter(
    col("co2_per_capita").isNotNull() & 
    col("renewables_share").isNotNull() &
    (col("Year") >= 2010) &
    (~col("Country").startswith("Total")) &
    (~col("Country").startswith("Other")) &
    (~col("Country").isin("Rest of World")) 
)
display(df_analysis)

## 6. Analytical Data Storage

After preparing the filtered dataset for analysis, the resulting DataFrame (`df_analysis`) was stored in Delta format as a dedicated analytical layer.

This step represents the creation of a refined, analysis-ready dataset derived from the transformed data. By storing this dataset separately, the pipeline adopts a layered architecture:

- **Raw layer**: original ingested data  
- **Processed layer**: cleaned and transformed dataset (`df_transformed`)  
- **Analytical layer**: filtered, high-quality dataset (`df_analysis`)  

This approach improves efficiency by avoiding repeated filtering operations and enables faster querying for visualisation and analysis tasks.

Additionally, storing the analytical dataset enhances reproducibility and supports modular pipeline design, where each stage can be independently accessed and reused.

Overall, this step reflects best practices in modern data engineering, where intermediate and final datasets are persisted to support scalable and maintainable workflows.

In [0]:
df_analysis.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/energy_schema/bdp_energy/energy_analysis"
)

## 7. Query and Visualisation

### 7.1 Regional Trends in CO₂ Emissions

To understand how carbon emissions vary globally, CO₂ emissions per capita were aggregated at the regional level over time. This allows for comparison of environmental impact across different parts of the world while accounting for population differences.

The following query computes the average CO₂ emissions per capita for each region by year:

In [0]:

display(
    df_analysis
    .filter(col("Region").isNotNull())
    .groupBy("Region", "Year")
    .avg("co2_per_capita")
    .orderBy("Region", "Year")
)

Databricks visualization. Run in Databricks to view.

#### Insight and Interpretation

The visualisation reveals several important patterns:

- **North America and the Middle East consistently exhibit the highest CO₂ emissions per capita**, reflecting high energy consumption levels and continued reliance on fossil fuels.  

- **Europe shows a gradual decline in emissions over time**, suggesting the impact of environmental policies, energy efficiency improvements, and increased adoption of renewable energy.  

- **Asia Pacific and CIS regions demonstrate moderate emissions**, with relatively stable trends, indicating growing industrial activity but also gradual shifts in energy sources.  

- **Africa maintains the lowest emissions per capita**, which may reflect lower industrialisation levels rather than sustainable energy practices.  

- A noticeable **dip around 2020** is observed across most regions, likely linked to reduced economic activity during the COVID-19 pandemic.

Overall, the chart highlights significant regional disparities in emissions and suggests that while some regions are transitioning towards lower emissions, others remain heavily dependent on carbon-intensive energy systems.

### 7.2 Global Distribution of Renewable Energy Adoption

To examine how renewable energy adoption varies across countries, a choropleth map was created using the most recent available data (2024). This provides a spatial perspective on how different regions are progressing in the transition towards cleaner energy sources.

The dataset was first filtered to include only observations from 2024 with valid renewable energy share values. The data was then converted to a Pandas DataFrame to enable visualisation using Plotly.


In [0]:
from pyspark.sql.functions import col
import plotly.express as px

# Step 1: Prepare clean data for map (latest year)
df_map = df_analysis.filter(
    (col("Year") == 2024) &
    (col("renewables_share").isNotNull()) 
   
).select(
    "Country",
    "renewables_share"
)

# Step 2: Convert to Pandas
pdf = df_map.toPandas()

# Step 3: Plot choropleth
fig = px.choropleth(
    pdf,
    locations="Country",
    locationmode="country names",
    color="renewables_share",
    color_continuous_scale="YlGn",
    title="Global Renewable Energy Share by Country (2024)"
)

fig.update_layout(
    title_x=0.5,
    geo=dict(showframe=False, showcoastlines=True)
)

fig.show()

#### Insight and Interpretation

The choropleth map highlights significant global variation in renewable energy adoption:

- **Countries in Europe, particularly in Northern and Western regions, show relatively high renewable energy shares**, reflecting strong policy support, investment in clean energy infrastructure, and long-term sustainability strategies.  

- **South America, particularly countries such as Brazil, also demonstrates high renewable energy usage**, largely due to reliance on hydroelectric power.  

- **Many countries in Africa and parts of Asia display lower renewable energy shares**, which may be attributed to limited infrastructure, investment constraints, or continued dependence on fossil fuels.  

- Some countries exhibit **very high renewable shares (approaching 1)**, indicating energy systems heavily dominated by renewables, although these may also reflect smaller or more specialised energy systems.

- The presence of missing regions (e.g., parts of Africa) suggests **data availability limitations**, highlighting a common challenge in global energy datasets.

Overall, the visualisation demonstrates that renewable energy adoption is unevenly distributed globally, with clear disparities between developed and developing regions. This reinforces the importance of policy, investment, and infrastructure in driving the global energy transition.

### 7.3 Relationship Between Renewable Energy and CO₂ Emissions

To investigate the relationship between renewable energy adoption and carbon emissions, a scatter plot was created using the most recent data (2024). Each point represents a country, with colour used to distinguish regions.

This allows for analysis of whether higher renewable energy usage is associated with lower CO₂ emissions per capita.

In [0]:
display(
    df_analysis
    .filter(col("Year") == 2024)
    .select("renewables_share", "co2_per_capita", "Region")
)

Databricks visualization. Run in Databricks to view.

#### Insight and Interpretation

The scatter plot reveals several important patterns:

- There is a **general negative relationship** between renewable energy share and CO₂ emissions per capita. Countries with higher renewable energy adoption tend to exhibit lower emissions, suggesting that increased reliance on clean energy contributes to reduced environmental impact.

- However, the relationship is **not perfectly linear**, indicating that renewable energy alone does not fully determine emissions levels. Other factors such as industrial activity, energy efficiency, and economic structure also play significant roles.

- **Clusters of countries with low renewable share (below 0.2)** show a wide range of emissions, highlighting that countries with similar energy mixes can still differ significantly in emissions intensity.

- **Outliers are present**, including countries with very high renewable shares but moderate emissions, and others with low renewable adoption but relatively low emissions. This suggests differences in energy demand, population size, or reliance on non-renewable but less carbon-intensive sources.

- Regional patterns are also visible:
  - **Middle East and North America** tend to have higher emissions with lower renewable shares  
  - **Europe** shows moderate renewable adoption with comparatively lower emissions  
  - **Africa** generally clusters at low emissions and low renewable share, reflecting lower overall energy consumption  

Overall, the analysis indicates that while renewable energy adoption is associated with lower emissions, it is not the sole determinant. A comprehensive transition towards sustainability requires a combination of increased renewable usage, improved efficiency, and structural changes in energy systems.

## Conclusion

This analysis demonstrates that while renewable energy adoption plays a key role in reducing carbon emissions, regional differences and structural factors significantly influence outcomes. The pipeline successfully transforms raw energy data into actionable insights, highlighting the importance of both data engineering and analytical reasoning in understanding global sustainability challenges.